<a href="https://colab.research.google.com/github/ekanursuci/uas-anvis/blob/main/dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install dash
!pip install dash-bootstrap-components
!pip install pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 6.9 MB/s eta 0:00:00


In [2]:
from google.colab import files

uploaded = files.upload()

Saving penjualan_mobil.csv to penjualan_mobil.csv


In [ ]:
import pandas as pd
import plotly.express as px

from dash import Dash
from dash import dcc
from dash import html
from dash import dash_table

from dash.dependencies import Input, Output, State

import dash_bootstrap_components as dbc

from pyngrok import ngrok
from threading import Thread
import time

# =====================================================
# BACA DATA
# =====================================================

df = pd.read_csv("penjualan_mobil.csv")
df.insert(0, "No", range(1, len(df) + 1))
# =====================================================
# KPI
# =====================================================

total_penjualan = int(df["Penjualan"].sum())
jumlah_kota = df["Kota"].nunique()
jumlah_merek = df["Merek"].nunique()

# =====================================================
# KOORDINAT KOTA
# =====================================================

koordinat = {
    "Jakarta": (-6.20,106.81),
    "Bandung": (-6.91,107.60),
    "Surabaya": (-7.25,112.75),
    "Medan": (3.59,98.67),
    "Palembang": (-2.99,104.76),
    "Makassar": (-5.13,119.41),
    "Semarang": (-6.99,110.42),
    "Yogyakarta": (-7.79,110.37),
    "Balikpapan": (-1.24,116.86),
    "Pekanbaru": (0.53,101.45)
}

# =====================================================
# LINE CHART
# =====================================================

trend_data = (
    df.groupby("Tahun")["Penjualan"]
    .sum()
    .reset_index()
)

trend_fig = px.line(
    trend_data,
    x="Tahun",
    y="Penjualan",
    markers=True,
    title="Trend Penjualan Mobil"
)

# =====================================================
# PIE CHART
# =====================================================

pie_data = (
    df.groupby("Merek")["Penjualan"]
    .sum()
    .reset_index()
)

pie_fig = px.pie(
    pie_data,
    names="Merek",
    values="Penjualan",
    title="Market Share Merek"
)

# =====================================================
# MAP
# =====================================================

sales_city = (
    df.groupby("Kota")["Penjualan"]
    .sum()
    .reset_index()
)

sales_city["Lat"] = sales_city["Kota"].map(
    lambda x: koordinat.get(x,(None,None))[0]
)

sales_city["Lon"] = sales_city["Kota"].map(
    lambda x: koordinat.get(x,(None,None))[1]
)

sales_city = sales_city.dropna()

map_fig = px.scatter_mapbox(
    sales_city,
    lat="Lat",
    lon="Lon",
    size="Penjualan",
    color="Penjualan",
    hover_name="Kota",
    zoom=3.5,
    center={"lat":-2,"lon":118},
    mapbox_style="open-street-map",
    title="Sebaran Penjualan Mobil Indonesia"
)

# =====================================================
# APP
# =====================================================

app = Dash(
    __name__,
    external_stylesheets=[dbc.themes.BOOTSTRAP],
    suppress_callback_exceptions=True
)

# =====================================================
# STYLE
# =====================================================

SIDEBAR_STYLE = {
    "position":"fixed",
    "top":"0",
    "left":"0",
    "bottom":"0",
    "width":"250px",
    "padding":"20px",
    "backgroundColor":"#212529",
    "color":"white",
    "transition":"all .3s"
}

CONTENT_STYLE = {
    "marginLeft":"270px",
    "padding":"20px",
    "transition":"all .3s"
}

# =====================================================
# SIDEBAR
# =====================================================

sidebar = html.Div(

    [

        html.H3(
            "🚗 PT Sejahtera Mobil",
            className="text-center"
        ),

        html.Hr(),

        dbc.Nav(

            [

                dbc.NavLink(
                    "📊 Dashboard",
                    href="/",
                    active="exact"
                ),

                dbc.NavLink(
                    "📋 Data Penjualan",
                    href="/data",
                    active="exact"
                )

            ],

            vertical=True,
            pills=True

        )

    ],

    id="sidebar",
    style=SIDEBAR_STYLE

)

# =====================================================
# NAVBAR
# =====================================================

navbar = dbc.Navbar(

    dbc.Container([

        dbc.Button(
            "☰",
            id="btn_sidebar",
            color="light"
        ),

        dbc.NavbarBrand(
            "Dashboard Penjualan Mobil Indonesia",
            className="ms-3"
        )

    ]),

    color="dark",
    dark=True

)

# =====================================================
# DASHBOARD PAGE
# =====================================================

dashboard_page = html.Div([

    dbc.Row([

        dbc.Col(

            dbc.Card([

                dbc.CardBody([

                    html.H5("Total Penjualan"),
                    html.H2(f"{total_penjualan:,}")

                ])

            ], color="primary", inverse=True),

            md=4

        ),

        dbc.Col(

            dbc.Card([

                dbc.CardBody([

                    html.H5("Jumlah Kota"),
                    html.H2(jumlah_kota)

                ])

            ], color="success", inverse=True),

            md=4

        ),

        dbc.Col(

            dbc.Card([

                dbc.CardBody([

                    html.H5("Jumlah Merek"),
                    html.H2(jumlah_merek)

                ])

            ], color="warning", inverse=True),

            md=4

        )

    ]),

    html.Br(),

    dbc.Row([

        dbc.Col(
            dcc.Graph(
                figure=trend_fig
            ),
            md=6
        ),

        dbc.Col(
            dcc.Graph(
                figure=pie_fig
            ),
            md=6
        )

    ]),

    html.Br(),

    dbc.Card([

        dbc.CardHeader(
            "Drill Down Penjualan"
        ),

        dbc.CardBody([

            html.P(
                "Klik batang merek untuk melihat detail kota"
            ),

            dcc.Graph(
                id="drilldown"
            )

        ])

    ]),

    html.Br(),

    dbc.Card([

        dbc.CardHeader(
            "Peta Sebaran Penjualan"
        ),

        dbc.CardBody(

            dcc.Graph(
                figure=map_fig
            )

        )

    ])

])

# =====================================================
# DATA PAGE
# =====================================================

data_page = html.Div([

    html.H3("Data Penjualan Mobil"),

    html.Hr(),

    dbc.Row([

        dbc.Col([

            html.Label("Filter Merek"),

            dcc.Dropdown(
                id="filter-merek",
                options=[
                    {"label": i, "value": i}
                    for i in sorted(df["Merek"].unique())
                ],
                placeholder="Pilih Merek",
                clearable=True
            )

        ], md=4),

        dbc.Col([

            html.Label("Filter Kota"),

            dcc.Dropdown(
                id="filter-kota",
                options=[
                    {"label": i, "value": i}
                    for i in sorted(df["Kota"].unique())
                ],
                placeholder="Pilih Kota",
                clearable=True
            )

        ], md=4),

        dbc.Col([

            html.Label("Filter Tahun"),

            dcc.Dropdown(
                id="filter-tahun",
                options=[
                    {"label": str(i), "value": i}
                    for i in sorted(df["Tahun"].unique())
                ],
                placeholder="Pilih Tahun",
                clearable=True
            )

        ], md=4)

    ]),

    html.Br(),

    html.H5(id="jumlah-data"),

    html.Br(),

    dash_table.DataTable(

        id="tabel-data",

        page_size=15,

        sort_action="native",

        style_table={
            "overflowX": "auto"
        },

        style_header={
            "fontWeight": "bold",
            "backgroundColor": "#f8f9fa"
        },

        style_cell={
            "textAlign": "center"
        }

    )

])

# =====================================================
# MAIN LAYOUT
# =====================================================

app.layout = html.Div([

    dcc.Location(id="url"),

    sidebar,

    html.Div([

        navbar,

        html.Br(),

        html.Div(
            id="page-content"
        )

    ],

    id="page-content-wrapper",

    style=CONTENT_STYLE)

])

# =====================================================
# ROUTING
# =====================================================

@app.callback(
    Output("page-content","children"),
    Input("url","pathname")
)
def display_page(pathname):

    if pathname == "/data":
        return data_page

    return dashboard_page

# =====================================================
# DRILL DOWN
# =====================================================

@app.callback(
    Output("drilldown","figure"),
    Input("drilldown","clickData")
)
def update_drill(clickData):

    if clickData is None:

        summary = (
            df.groupby("Merek")["Penjualan"]
            .sum()
            .reset_index()
        )

        return px.bar(
            summary,
            x="Merek",
            y="Penjualan",
            title="Klik Merek Untuk Melihat Detail Kota"
        )

    merek = clickData["points"][0]["x"]

    detail = (
        df[df["Merek"] == merek]
        .groupby("Kota")["Penjualan"]
        .sum()
        .reset_index()
    )

    return px.bar(
        detail,
        x="Kota",
        y="Penjualan",
        title=f"Penjualan {merek} per Kota"
    )

# =====================================================
# HAMBURGER MENU
# =====================================================

@app.callback(
    [
        Output("sidebar","style"),
        Output("page-content-wrapper","style")
    ],
    Input("btn_sidebar","n_clicks")
)
def toggle_sidebar(n):

    if n and n % 2 == 1:

        sidebar_hidden = SIDEBAR_STYLE.copy()
        sidebar_hidden["left"] = "-250px"

        content_expand = CONTENT_STYLE.copy()
        content_expand["marginLeft"] = "20px"

        return sidebar_hidden, content_expand

    return SIDEBAR_STYLE, CONTENT_STYLE

    # =====================================================
# FILTER DATA PENJUALAN
# =====================================================

@app.callback(

    [
        Output("tabel-data", "data"),
        Output("tabel-data", "columns"),
        Output("jumlah-data", "children")
    ],

    [
        Input("filter-merek", "value"),
        Input("filter-kota", "value"),
        Input("filter-tahun", "value")
    ]

)
def filter_data(merek, kota, tahun):

    dff = df.copy()

    if merek:
        dff = dff[dff["Merek"] == merek]

    if kota:
        dff = dff[dff["Kota"] == kota]

    if tahun:
        dff = dff[dff["Tahun"] == tahun]

    columns = [
        {"name": i, "id": i}
        for i in dff.columns
    ]

    info = (
        f"Menampilkan {len(dff)} data "
        f"dari total {len(df)} data"
    )

    return (
        dff.to_dict("records"),
        columns,
        info
    )

# =====================================================
# NGROK
# =====================================================

PORT = 8050

ngrok.kill()

# GANTI DENGAN TOKEN NGROK MILIK ANDA
ngrok.set_auth_token("3Fsqu9dSlqceSbnRG8QCVXu9mCD_6n61vKH5zFYuW6jUCEDEi")

public_url = ngrok.connect(PORT)

print("="*60)
print("URL DASHBOARD")
print(public_url.public_url)
print("="*60)

# =====================================================
# RUN DASH
# =====================================================

app.run(
    host="0.0.0.0",
    port=PORT,
    debug=False
)

URL DASHBOARD
https://construct-scarring-unruffled.ngrok-free.dev
Dash is running on http://0.0.0.0:8050/



INFO:dash.dash:Dash is running on http://0.0.0.0:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8050
 * Running on http://172.28.0.12:8050
INFO:werkzeug:Press CTRL+C to quit
